In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchattacks
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.spatial.distance import mahalanobis
from scipy.linalg import inv
import warnings
warnings.filterwarnings('ignore')

# ----------------------------
# 1. Настройки
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
BATCH_SIZE = 64
EPOCHS_AT = 5
LR = 0.001
EPSILONS = [0.0020, 0.0050, 0.0100, 0.0300, 0.0500]
BIM_STEPS = 10
print(f"🖥️ Device: {device}")

# ----------------------------
# 2. Данные
# ----------------------------
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
trainset = torchvision.datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
testset  = torchvision.datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
testloader  = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ----------------------------
# 3. Модели
# ----------------------------
class Model_A(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 64, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool2d(2)
        )
        with torch.no_grad():
            self.feat_size = self.features(torch.zeros(1, 1, 28, 28)).numel()
        self.fc1 = nn.Linear(self.feat_size, 256)
        self.fc2 = nn.Linear(256, 10)
        self.relu = nn.ReLU()
        self.feature_maps = {}

    def forward(self, x):
        x = self.features(x)
        self.feature_maps['layer2'] = x
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class Model_B(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        with torch.no_grad():
            self.feat_size = self.features(torch.zeros(1, 1, 28, 28)).numel()
        self.fc1 = nn.Linear(self.feat_size, 512)
        self.fc2 = nn.Linear(512, 10)
        self.relu = nn.ReLU()
        self.feature_maps = {}

    def forward(self, x):
        x = self.features(x)
        self.feature_maps['layer2'] = x
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ----------------------------
# 4. Адверсариальное обучение (PGD-AT)
# ----------------------------
def train_adversarial(model, loader, epochs, lr, eps_at=0.2, alpha_at=0.02, steps_at=7):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    attack = torchattacks.PGD(model, eps=eps_at, alpha=alpha_at, steps=steps_at, random_start=True)
    
    for epoch in range(epochs):
        loss_c, loss_a = 0.0, 0.0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss_clean = criterion(model(X), y)
            X_adv = attack(X, y)
            loss_adv = criterion(model(X_adv), y)
            loss = 0.5 * loss_clean + 0.5 * loss_adv
            loss.backward()
            optimizer.step()
            loss_c += loss_clean.item()
            loss_a += loss_adv.item()
        print(f"  AT Epoch {epoch+1}/{epochs} | L_clean: {loss_c/len(loader):.4f} | L_adv: {loss_a/len(loader):.4f}")
    return model

# ----------------------------
# 5. Извлечение признаков
# ----------------------------
def extract_features_and_labels(model, loader):
    model.eval()
    feats, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            _ = model(X)
            feats.append(model.feature_maps['layer2'].cpu().numpy().reshape(X.size(0), -1))
            labels.append(y.numpy().astype(int))
    return np.vstack(feats), np.concatenate(labels)

# ----------------------------
# 6. Детектор Махаланобиса
# ----------------------------
class MahalanobisDetector:
    def __init__(self, reg_cov=1e-4):
        self.class_means = {}
        self.inv_covs = {}
        self.reg = reg_cov
    def fit(self, X, y, num_classes=10):
        for c in range(num_classes):
            mask = y == c
            feats = X[mask]
            if len(feats) == 0: continue
            self.class_means[c] = feats.mean(axis=0)
            cov = np.cov(feats, rowvar=False)
            cov += self.reg * np.eye(cov.shape[0])
            self.inv_covs[c] = inv(cov)
        return self
    def score(self, X, y):
        scores = []
        for i in range(len(y)):
            c = int(y[i])
            if c not in self.class_means:
                scores.append(1e6); continue
            diff = X[i] - self.class_means[c]
            d = mahalanobis(diff, np.zeros_like(diff), self.inv_covs[c])
            scores.append(d**2 if np.isfinite(d) else 1e6)
        return np.array(scores)

# ----------------------------
# 7. Метрики
# ----------------------------
def tpr_at_fpr(y_true, y_scores, fpr_target):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    return np.interp(fpr_target, fpr, tpr)

# ----------------------------
# 8. Запуск
# ----------------------------
if __name__ == "__main__":
    models = {"Model_A_AT": Model_A, "Model_B_AT": Model_B}

    print(f"\n🚀 Оценка обнаруживаемости (атака: BIM, шаги={BIM_STEPS}, alpha=eps/4, детектор: Махаланобис)")
    print(f"{'Модель':<12} | {'ε':<8} | {'AUC':<7} | {'TPR@1%':<7} | {'TPR@5%':<7} | {'TPR@10%':<7}")
    print("-" * 65)

    for model_name, model_cls in models.items():
        print(f"\n  [Обучение] {model_name} (PGD-AT, eps=0.2)...")
        model = model_cls().to(device)
        train_adversarial(model, trainloader, EPOCHS_AT, LR, eps_at=0.2, alpha_at=0.02, steps_at=7)

        print(f"  [Извлечение чистых активаций и обучение детектора]...")
        X_clean, y_clean = extract_features_and_labels(model, testloader)
        detector = MahalanobisDetector(reg_cov=1e-4).fit(X_clean, y_clean)
        scores_clean = detector.score(X_clean, y_clean)

        for eps in EPSILONS:
            model.eval()
            X_adv_list, y_adv_list = [], []
            atk = torchattacks.BIM(model, eps=eps, steps=BIM_STEPS, alpha=eps/4)
            
            for X, y in testloader:
                X, y = X.to(device), y.to(device)
                X_adv = atk(X, y)  # Генерация атаки (требует градиенты)
                with torch.no_grad():
                    _ = model(X_adv)
                    f = model.feature_maps['layer2'].cpu().numpy().reshape(X_adv.size(0), -1)
                X_adv_list.append(f)
                y_adv_list.append(y.cpu().numpy().astype(int))
                
            X_adv = np.vstack(X_adv_list)
            y_adv = np.concatenate(y_adv_list)
            scores_adv = detector.score(X_adv, y_adv)

            # Локальная коррекция направления скоров (на случай инверсии)
            sc_c = scores_clean.copy()
            sc_a = scores_adv.copy()
            if np.mean(sc_c) > np.mean(sc_a):
                sc_c = -sc_c
                sc_a = -sc_a
                
            y_true = np.concatenate([np.zeros(len(sc_c)), np.ones(len(sc_a))])
            y_scores = np.concatenate([sc_c, sc_a])

            auc = roc_auc_score(y_true, y_scores)
            tpr1 = tpr_at_fpr(y_true, y_scores, 0.01)
            tpr5 = tpr_at_fpr(y_true, y_scores, 0.05)
            tpr10 = tpr_at_fpr(y_true, y_scores, 0.10)
            
            print(f"  {model_name:<12} | {eps:<8.4f} | {auc:<7.4f} | {tpr1:<7.3f} | {tpr5:<7.3f} | {tpr10:<7.3f}")

🖥️ Device: cpu

🚀 Оценка обнаруживаемости (атака: BIM, шаги=10, alpha=eps/4, детектор: Махаланобис)
Модель       | ε        | AUC     | TPR@1%  | TPR@5%  | TPR@10%
-----------------------------------------------------------------

  [Обучение] Model_A_AT (PGD-AT, eps=0.2)...
  AT Epoch 1/5 | L_clean: 0.4694 | L_adv: 1.0247
  AT Epoch 2/5 | L_clean: 0.3028 | L_adv: 0.8172
  AT Epoch 3/5 | L_clean: 0.2592 | L_adv: 0.7603
  AT Epoch 4/5 | L_clean: 0.2310 | L_adv: 0.7293
  AT Epoch 5/5 | L_clean: 0.2073 | L_adv: 0.7043
  [Извлечение чистых активаций и обучение детектора]...
  Model_A_AT   | 0.0020   | 0.6411  | 0.357   | 0.390   | 0.415  
  Model_A_AT   | 0.0050   | 0.8620  | 0.799   | 0.806   | 0.811  
  Model_A_AT   | 0.0100   | 0.9620  | 0.950   | 0.951   | 0.952  
  Model_A_AT   | 0.0300   | 1.0000  | 1.000   | 1.000   | 1.000  
  Model_A_AT   | 0.0500   | 1.0000  | 1.000   | 1.000   | 1.000  

  [Обучение] Model_B_AT (PGD-AT, eps=0.2)...
  AT Epoch 1/5 | L_clean: 0.4358 | L_adv: 0.979